In [53]:
import numpy as np 
import pandas as pd 

import os

In [54]:
import json
import os
import pandas as pd
import numpy as np

DATA_DIR = '../input/competitions/cassava-leaf-disease-classification/'

with open(os.path.join(DATA_DIR, 'label_num_to_disease_map.json')) as f:
    disease_map = json.load(f)

df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))

# 숫자로 된 label을 실제 질병 이름으로 바꾼 열을 하나 더 추가
df['disease_name'] = df['label'].astype(str).map(disease_map)

In [55]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import WeightedRandomSampler


train_transform = A.Compose([
    A.RandomResizedCrop(size=(256,256), p=1.0), # 확률 100%로 사이즈 512*512으로 무작위 확대 및 자르기
    A.HorizontalFlip(p=0.5),      # 좌우 반전
    A.VerticalFlip(p=0.5),        # 상하 반전
    A.ShiftScaleRotate(rotate_limit=20, p=0.5), # -20~+20도 사이 랜덤 회전으로 각도 다양성 확보

    # 색감 변화
    A.HueSaturationValue(hue_shift_limit=0.2, sat_shift_limit=0.2, val_shift_limit=0.2, p=0.5), # 색상, 채도, 명도를 최대 20% 범위 내에서 변경

    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # 정규화
    ToTensorV2() # Tensor 변환
])


test_transform = A.Compose([
    A.Resize(256,256),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [56]:
from sklearn.model_selection import train_test_split


train_df, test_df = train_test_split(df, test_size=0.2, random_state=42,
                                     stratify=df['label']) #클래스 불균형이 발생했으므로 label의 개수를 동일하게 split

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"학습용 데이터 개수: {len(train_df)}장")
print(f"테스트용 데이터 개수: {len(test_df)}장")

학습용 데이터 개수: 17117장
테스트용 데이터 개수: 4280장


In [57]:
import torch
from torch.utils.data import Dataset
import cv2

class CassavaDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)


    def __getitem__(self, index):
      image_name = self.df.loc[index, 'image_id']
      label = self.df.loc[index, 'label']

      image_path = os.path.join(self.image_dir, image_name)
      image = cv2.imread(image_path)
      image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

      if self.transform is not None:
          augmented = self.transform(image=image)
          image = augmented['image']

      return image, torch.tensor(label, dtype=torch.long)

In [58]:
from torch.utils.data import DataLoader

train_dataset = CassavaDataset(df=train_df, image_dir=DATA_DIR+'train_images', transform=train_transform)
test_dataset = CassavaDataset(df=test_df, image_dir=DATA_DIR+'train_images', transform=test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

# 모델 변경
# ResNet > EfficientNet

In [59]:
from torchvision import models
import torch.nn as nn
import timm

model = timm.create_model('tf_efficientnet_b4_ns', pretrained=True, num_classes=5)

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  model = create_fn(


In [60]:
# 모델 GPU로 전송

if torch.cuda.is_available():
    print("yes")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

yes


In [61]:
#손실함수와 옵티마이저 설정

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-6)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-6)

In [62]:
from torch import amp 


epochs = 20
scaler = amp.GradScaler('cuda')

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        # 1. 자동 혼합 정밀도(AMP) 적용
        with amp.autocast('cuda'): 
            outputs = model(images)
            loss = criterion(outputs, labels)
    
        # 2. 역전파 및 가중치 업데이트 
        scaler.scale(loss).backward() 
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        
    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] 완료! - Loss: {epoch_loss:.4f}")
    scheduler.step()

Epoch [1/20] 완료! - Loss: 1.0141
Epoch [2/20] 완료! - Loss: 0.6570
Epoch [3/20] 완료! - Loss: 0.5705
Epoch [4/20] 완료! - Loss: 0.5151
Epoch [5/20] 완료! - Loss: 0.4840
Epoch [6/20] 완료! - Loss: 0.4530
Epoch [7/20] 완료! - Loss: 0.4321
Epoch [8/20] 완료! - Loss: 0.4083
Epoch [9/20] 완료! - Loss: 0.3936
Epoch [10/20] 완료! - Loss: 0.3777
Epoch [11/20] 완료! - Loss: 0.3627
Epoch [12/20] 완료! - Loss: 0.3476
Epoch [13/20] 완료! - Loss: 0.3366
Epoch [14/20] 완료! - Loss: 0.3235
Epoch [15/20] 완료! - Loss: 0.3101
Epoch [16/20] 완료! - Loss: 0.3096
Epoch [17/20] 완료! - Loss: 0.3045
Epoch [18/20] 완료! - Loss: 0.2959
Epoch [19/20] 완료! - Loss: 0.2973
Epoch [20/20] 완료! - Loss: 0.2890


In [64]:
from sklearn.metrics import accuracy_score, classification_report

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

final_accuracy = accuracy_score(all_labels, all_preds)
print(f"/n Test 정확도: {final_accuracy * 100:.2f}%")

print("\n Classification Report")

disease_names = ['CBB (0)', 'CBSD (1)', 'CGM (2)', 'CMD (3)', 'Healthy (4)']
print(classification_report(all_labels, all_preds, target_names=disease_names))

/n Test 정확도: 84.14%

 Classification Report
              precision    recall  f1-score   support

     CBB (0)       0.57      0.55      0.56       217
    CBSD (1)       0.79      0.66      0.72       438
     CGM (2)       0.73      0.71      0.72       477
     CMD (3)       0.93      0.95      0.94      2632
 Healthy (4)       0.63      0.67      0.65       516

    accuracy                           0.84      4280
   macro avg       0.73      0.71      0.72      4280
weighted avg       0.84      0.84      0.84      4280

